STEP 6: IMAGE CLASSIFICATION -> CNN TRAINING AND  CHOOSING THE BEST MODEL

In [5]:
# open the csv file containing the data for the downloaded women images 
import pandas as pd
df_images = pd.read_csv("data\\image_dataset_for_modelling.csv")

print(df_images.shape)
print(df_images.columns)
print(df_images.head())

(10191, 19)
Index(['id', 'product_display_name', 'brand_name', 'price', 'discounted_price',
       'myntra_rating', 'age_group', 'gender', 'master_category',
       'sub_category', 'article_type', 'base_colour', 'usage', 'season',
       'year', 'display_categories', 'image_url', 'filename', 'link'],
      dtype='object')
      id                     product_display_name brand_name   price  \
0  10003   Nike Women As Nike Eleme White T-Shirt       Nike  2695.0   
1  10007  Nike Women As Trophy Swo White T-Shirts       Nike  1095.0   
2  10019    Nike Women As Miler Ss White T-Shirts       Nike  1295.0   
3  10024  Nike Women As Make Yours Black T-Shirts       Nike   995.0   
4  10025    Nike Women As Element Ja Pink T-Shirt       Nike  2995.0   

   discounted_price  myntra_rating     age_group gender master_category  \
0            2695.0              1  Adults-Women  Women         Apparel   
1            1095.0              1  Adults-Women  Women         Apparel   
2            1295.

IMPLEMENTATION OF TRANSFER LEARNING -> USING PRE-TRAINED CNN MODELS SUCH AS RESNET50, MOBILENETV2, EFFCIENTNETB0

In [6]:
# common imports

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [7]:
# split  dataset

train_df, temp_df = train_test_split(
    df_images, test_size=0.3, stratify=df_images['article_type'], 
    random_state=42)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['article_type'], 
    random_state=42)

In [8]:
# MOBILENETV2

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# image preprocessing  and data  augmentation

# load imagedatagenerator

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    zoom_range = 0.2,
    rotation_range = 20,
    horizontal_flip = True
)

val_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

test_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

train_data = train_datagen.flow_from_dataframe(

    dataframe = train_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical"
)

val_data = val_datagen.flow_from_dataframe(

    dataframe = val_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical"
)

test_data = test_datagen.flow_from_dataframe(

    dataframe = test_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical",
    shuffle = False
)

# load pretrained model mobilenetv2
base_model1 = MobileNetV2(weights = "imagenet", include_top = False, input_shape = (224,224,3))

# fine -tuning layers


# freeze all layers 
for layer in base_model1.layers:
    layer.trainable = False

# add custom classification head

x = base_model1.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(10, activation="softmax")(x)

mobilenetv2_model = Model(inputs = base_model1.input, outputs = output)

# compile the model

mobilenetv2_model.compile(
    optimizer = 'adam',
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)
# train the  model
mobilenetv2_model.fit(train_data, validation_data= val_data, epochs = 5, callbacks=[early_stop])


# 2. fine tuning
for layer in base_model1.layers[:-30]:
    layer.trainable = False

# tune  last 30 layers
for layer in base_model1.layers[-30:]:
    layer.trainable = True

mobilenetv2_model.compile(
    optimizer = Adam(0.0001),
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

mobilenetv2_model.fit(train_data, validation_data= val_data, epochs = 5, callbacks=[early_stop])


#print summary
print(mobilenetv2_model.summary())

test_data.reset()

#evaluate model
test_loss_mobilenetv2, test_acc_mobilenetv2 = mobilenetv2_model.evaluate(test_data)
print("mobilenetv2 test accuracy:", test_acc_mobilenetv2)

# predictions
mobilenetv2_predictions = mobilenetv2_model.predict(test_data)
mobilenetv2_predicted_classes = np.argmax(mobilenetv2_predictions, axis=1)
true_classes = test_data.classes

class_labels = list(test_data.class_indices.keys())

mobilenetv2_cm = confusion_matrix(true_classes, mobilenetv2_predicted_classes)
print("mobilenetv2 confusion matrix: ", mobilenetv2_cm)

mobilenetv2_cr = classification_report(true_classes, mobilenetv2_predicted_classes, target_names=class_labels)
print("mobilenetv2  classification report: ", mobilenetv2_cr)








Found 7133 validated image filenames belonging to 10 classes.
Found 1529 validated image filenames belonging to 10 classes.
Found 1529 validated image filenames belonging to 10 classes.



Epoch 1/5


223/223 [==============================] - 633s 3s/step - loss: 0.4770 - accuracy: 0.8116 - val_loss: 0.2938 - val_accuracy: 0.8764
Epoch 2/5
223/223 [==============================] - 395s 2s/step - loss: 0.3215 - accuracy: 0.8628 - val_loss: 0.2640 - val_accuracy: 0.8842
Epoch 3/5
223/223 [==============================] - 657s 3s/step - loss: 0.2862 - accuracy: 0.8775 - val_loss: 0.2769 - val_accuracy: 0.8757
Epoch 4/5
223/223 [==============================] - 406s 2s/step - loss: 0.2751 - accuracy: 0.8822 - val_loss: 0.2498 - val_accuracy: 0.8999
Epoch 5/5
223/223 [==============================] - 423s 2s/step - loss: 0.2533 - accuracy: 0.8876 - val_loss: 0.2505 - val_accuracy: 0.8901
Epoch 1/5
223/223 [==============================] - 473s 2s/step - loss: 0.3085 - accuracy: 0.8713

In [9]:
import tensorflow as tf
from tensorflow.keras import backend as K
K.clear_session()

In [10]:
# RESNET50

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# image preprocessing  and data  augmentation

# load imagedatagenerator

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    zoom_range = 0.2,
    rotation_range = 20,
    horizontal_flip = True
)

val_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

test_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

train_data = train_datagen.flow_from_dataframe(

    dataframe = train_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical"
)

val_data = val_datagen.flow_from_dataframe(

    dataframe = val_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical"
)

test_data = test_datagen.flow_from_dataframe(

    dataframe = test_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical",
    shuffle = False
)

# load pretrained model mobilenetv2
base_model2 = ResNet50(weights = "imagenet", include_top = False, input_shape = (224,224,3))

# fine -tuning layers


# freeze all layers 
for layer in base_model2.layers:
    layer.trainable = False

# add custom classification head

x = base_model2.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(10, activation="softmax")(x)

resnet_model = Model(inputs = base_model2.input, outputs = output)

# compile the model

resnet_model.compile(
    optimizer = 'adam',
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)
# train the  model
resnet_model.fit(train_data, validation_data= val_data, epochs = 5, callbacks=[early_stop])


# 2. fine tuning
for layer in base_model2.layers[:-30]:
    layer.trainable = False

# tune  last 30 layers
for layer in base_model2.layers[-30:]:
    layer.trainable = True

resnet_model.compile(
    optimizer = Adam(0.0001),
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

resnet_model.fit(train_data, validation_data= val_data, epochs = 5, callbacks=[early_stop])


#print summary
print(resnet_model.summary())

test_data.reset()

#evaluate model
test_loss_resnet, test_acc_resnet = resnet_model.evaluate(test_data)
print("resnet50 test accuracy:", test_acc_resnet)

# predictions
resnet_predictions = resnet_model.predict(test_data)
resnet_predicted_classes = np.argmax(resnet_predictions, axis=1)
true_classes = test_data.classes

class_labels = list(test_data.class_indices.keys())

resnet_cm = confusion_matrix(true_classes, resnet_predicted_classes)
print("resnet50 confusion matrix: ", resnet_cm)

resnet_cr = classification_report(true_classes, resnet_predicted_classes, target_names=class_labels)
print("resnet50 classification report: ", resnet_cr)








Found 7133 validated image filenames belonging to 10 classes.
Found 1529 validated image filenames belonging to 10 classes.
Found 1529 validated image filenames belonging to 10 classes.
Epoch 1/5
223/223 [==============================] - 803s 4s/step - loss: 0.4863 - accuracy: 0.8081 - val_loss: 0.2958 - val_accuracy: 0.8640
Epoch 2/5
223/223 [==============================] - 823s 4s/step - loss: 0.3168 - accuracy: 0.8604 - val_loss: 0.2658 - val_accuracy: 0.8888
Epoch 3/5
223/223 [==============================] - 838s 4s/step - loss: 0.2828 - accuracy: 0.8811 - val_loss: 0.2631 - val_accuracy: 0.8757
Epoch 4/5
223/223 [==============================] - 843s 4s/step - loss: 0.2703 - accuracy: 0.8831 - val_loss: 0.2563 - val_accuracy: 0.8934
Epoch 5/5
223/223 [==============================] - 831s 4s/step - loss: 0.2594 - accuracy: 0.8856 - val_loss: 0.2527 - val_accuracy: 0.8908
Epoch 1/5
223/223 [==============================] - 1060s 5s/step - loss: 0.2749 - accuracy: 0.8856 - v

In [11]:
import tensorflow as tf
from tensorflow.keras import backend as K
K.clear_session()

In [12]:
# EFFICIENTNETB0

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# image preprocessing  and data  augmentation

# load imagedatagenerator

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    zoom_range = 0.2,
    rotation_range = 20,
    horizontal_flip = True
)

val_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

test_datagen = ImageDataGenerator(preprocessing_function = preprocess_input)

train_data = train_datagen.flow_from_dataframe(

    dataframe = train_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical"
)

val_data = val_datagen.flow_from_dataframe(

    dataframe = val_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical"
)

test_data = test_datagen.flow_from_dataframe(

    dataframe = test_df,
    directory = "women_images",
    x_col = "filename",
    y_col = "article_type",
    target_size  = (224,224),
    batch_size = 32,
    class_mode = "categorical",
    shuffle = False
)

# load pretrained model mobilenetv2
base_model3 = EfficientNetB0(weights = "imagenet", include_top = False, input_shape = (224,224,3))

# fine -tuning layers


# freeze all layers 
for layer in base_model3.layers:
    layer.trainable = False

# add custom classification head

x = base_model3.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

output = Dense(10, activation="softmax")(x)

effnet_model = Model(inputs = base_model3.input, outputs = output)

# compile the model

effnet_model.compile(
    optimizer = 'adam',
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)
# train the  model
effnet_model.fit(train_data, validation_data= val_data, epochs = 5, callbacks=[early_stop])


# 2. fine tuning
for layer in base_model3.layers[:-50]:
    layer.trainable = False

# tune  last 30 layers
for layer in base_model3.layers[-50:]:
    layer.trainable = True

effnet_model.compile(
    optimizer = Adam(0.0001),
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

effnet_model.fit(train_data, validation_data= val_data, epochs = 5, callbacks=[early_stop])


#print summary
print(effnet_model.summary())

test_data.reset()

#evaluate model
test_loss_effnet, test_acc_effnet = effnet_model.evaluate(test_data)
print("effnetb0 test accuracy:", test_acc_effnet)

# predictions
effnet_predictions = effnet_model.predict(test_data)
effnet_predicted_classes = np.argmax(effnet_predictions, axis=1)
true_classes = test_data.classes

class_labels = list(test_data.class_indices.keys())

effnet_cm = confusion_matrix(true_classes, effnet_predicted_classes)
print("efficientnetb0 confusion matrix: ", effnet_cm)

effnet_cr = classification_report(true_classes, effnet_predicted_classes, target_names=class_labels)
print("efficientnetb0 classification report: ", effnet_cr)








Found 7133 validated image filenames belonging to 10 classes.
Found 1529 validated image filenames belonging to 10 classes.
Found 1529 validated image filenames belonging to 10 classes.
Epoch 1/5
223/223 [==============================] - 555s 2s/step - loss: 0.4252 - accuracy: 0.8284 - val_loss: 0.2589 - val_accuracy: 0.8882
Epoch 2/5
223/223 [==============================] - 516s 2s/step - loss: 0.2798 - accuracy: 0.8765 - val_loss: 0.2474 - val_accuracy: 0.8842
Epoch 3/5
223/223 [==============================] - 507s 2s/step - loss: 0.2490 - accuracy: 0.8891 - val_loss: 0.2441 - val_accuracy: 0.8927
Epoch 4/5
223/223 [==============================] - 507s 2s/step - loss: 0.2341 - accuracy: 0.8943 - val_loss: 0.2297 - val_accuracy: 0.9019
Epoch 5/5
223/223 [==============================] - 705s 3s/step - loss: 0.2150 - accuracy: 0.9062 - val_loss: 0.2355 - val_accuracy: 0.8960
Epoch 1/5
223/223 [==============================] - 556s 2s/step - loss: 0.2654 - accuracy: 0.8936 - va

INEFERENCE:  OF ALL MODELS PROPOSED, EFFICIENT NET GIVES THE BEST PRECISION AND RECALL BALANCE, FASTER AND STILL EFFICIENT WITH GOOD ACCUARCY . HENCE CHOSEN AS FINAL MODEL FOR IMAGE  CLASSIFICATION AND SIMILARITY COMPUTATION.

In [13]:
effnet_model.save("efficientnet_model.keras")